In [ ]:
import os
import pandas as pd
import multiprocessing
from time import time as timer
from pathlib import Path
from functools import partial
import urllib.request
import urllib.parse
import json

# ============ SUPPRESS TOKENIZERS WARNING ============
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# --- Helper Functions for Downloading ---
def download_image(image_link, savefolder):
    """
    Downloads a single image from a URL.
    Creates a valid filename from the URL to avoid errors.
    """
    if isinstance(image_link, str):
        try:
            parsed_url = urllib.parse.urlparse(image_link)
            filename = Path(urllib.parse.unquote(parsed_url.path)).name
            if not filename:
                return  # Skip if no valid filename can be found
            image_save_path = os.path.join(savefolder, filename)
            if not os.path.exists(image_save_path):
                urllib.request.urlretrieve(image_link, image_save_path)
        except Exception:
            # Errors are ignored to prevent the script from stopping.
            pass
    return

def download_images_parallel(image_links, download_folder):
    """
    Downloads a list of images in parallel using all available CPU cores.
    """
    if not os.path.exists(download_folder):
        os.makedirs(download_folder)
    
    num_processes = multiprocessing.cpu_count()
    print(f"🚀 Using {num_processes} CPU cores to download images...")
    
    download_func_partial = partial(download_image, savefolder=download_folder)
    
    with multiprocessing.Pool(num_processes) as pool:
        pool.map(download_func_partial, image_links)

# --- Main Script ---
# 1. DEFINE FILE PATHS
KAGGLE_OUTPUT_DIR = '/kaggle/working/image_dataset_for_kaggle'
IMAGES_DIR = os.path.join(KAGGLE_OUTPUT_DIR, 'images')
METADATA_FILE = os.path.join(KAGGLE_OUTPUT_DIR, 'dataset-metadata.json')
CSV_FILE_PATH = '/kaggle/input/train-sample-dataset/train.csv' 
SAMPLED_CSV_PATH = os.path.join(KAGGLE_OUTPUT_DIR, 'sampled_data.csv')

# 2. READ THE CSV FILE
try:
    df = pd.read_csv(CSV_FILE_PATH)
    print(f"✅ Successfully read '{CSV_FILE_PATH}'.")
except FileNotFoundError:
    print(f"❌ Error: Could not find '{CSV_FILE_PATH}'.")
    print("Please ensure you have uploaded the file and the name is correct.")
    exit()

# 3. FILTER AND SAMPLE DATA
initial_rows = len(df)
df = df[df['sample_id'] != 279285]
filtered_rows = len(df)
print(f"Removed row with sample_id == 279285. Remaining rows: {filtered_rows}/{initial_rows}")

# Randomly sample 1000 rows
sample_df = df.sample(n=1000, random_state=42)

# Ensure output directory exists before saving
os.makedirs(KAGGLE_OUTPUT_DIR, exist_ok=True)

# Save the sampled dataset
sample_df.to_csv(SAMPLED_CSV_PATH, index=False)
print(f"✅ Created a random sample of 1000 rows and saved to '{SAMPLED_CSV_PATH}'")

# 4. DOWNLOAD IMAGES
image_links = sample_df['image_link'].dropna().unique().tolist()
print(f"Found {len(image_links)} unique image links to download from the sample.")

start_time = timer()
download_images_parallel(image_links, IMAGES_DIR)
end_time = timer()

print(f"\n✅ Finished downloading images to '{IMAGES_DIR}'.")
print(f"   Total download time: {end_time - start_time:.2f} seconds.")

# 5. CREATE KAGGLE METADATA FILE
metadata = {
  "title": "My Awesome Image Dataset (Sampled 1000)",
  "id": "sourabhkap/your-dataset-name",
  "licenses": [
    {"name": "CC0-1.0"}
  ]
}

if not os.path.exists(KAGGLE_OUTPUT_DIR):
    os.makedirs(KAGGLE_OUTPUT_DIR)

with open(METADATA_FILE, 'w') as f:
    json.dump(metadata, f, indent=4)

print(f"✅ Successfully created 'dataset-metadata.json'.")

# 6. FINAL VERIFICATION
try:
    downloaded_count = len(os.listdir(IMAGES_DIR))
    print(f"   Downloaded {downloaded_count} images.")
except FileNotFoundError:
    downloaded_count = 0
    print("   No images were downloaded.")

print("\n🎉 All done! Your 1000-sample dataset is ready in the output directory.")

In [ ]:
import os
import warnings
# Suppress TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
# Suppress other warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from transformers import AutoImageProcessor, LlamaTokenizerFast
from huggingface_hub import hf_hub_download
# Paths
CSV_FILE_PATH = '/kaggle/working/image_dataset_for_kaggle/sampled_data.csv'
MODEL_NAME = "google/siglip-base-patch16-224"
# Download tokenizer files directly
print("Downloading tokenizer files...")
tokenizer_json = hf_hub_download(repo_id=MODEL_NAME, filename="tokenizer.json")
tokenizer_config = hf_hub_download(repo_id=MODEL_NAME, filename="tokenizer_config.json")
special_tokens = hf_hub_download(repo_id=MODEL_NAME, filename="special_tokens_map.json")
# Load tokenizer directly from files (bypasses the chat template check)
tokenizer = LlamaTokenizerFast(
    tokenizer_file=tokenizer_json,
    padding_side="right",
    model_max_length=64
)

# Set the pad token (critical for padding)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

print("✅ Tokenizer loaded successfully!")
# Load image processor
image_processor = AutoImageProcessor.from_pretrained(MODEL_NAME, use_fast=True)
print("✅ Image processor loaded successfully!")
# Load data
df = pd.read_csv(CSV_FILE_PATH)
print(f"Loaded {len(df)} samples")
# Process text with the tokenizer
texts = df['catalog_content'].astype(str).tolist()
inputs_text = tokenizer(
    texts,
    padding="max_length",
    truncation=True,
    max_length=64,
    return_tensors="np"
)
print(f"Text preprocessing complete:")
print(f"  input_ids shape: {inputs_text['input_ids'].shape}")
print(f"  attention_mask shape: {inputs_text['attention_mask'].shape}")
# Save text features
np.save('/kaggle/working/text_input_ids.npy', inputs_text['input_ids'])
np.save('/kaggle/working/text_attention_mask.npy', inputs_text['attention_mask'])
print("✅ Text features saved!")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
import os
import urllib.parse
from pathlib import Path

# ============ CONFIGURATION ============
IMAGE_FOLDER = '/kaggle/working/image_dataset_for_kaggle/images'
BATCH_SIZE = 64  # Adjust based on memory (can go higher with T4)
NUM_WORKERS = 4

print(f"\n🖼️  Starting image preprocessing...")
print(f"Processing {len(df)} images from {IMAGE_FOLDER}")

# ============ CUSTOM COLLATE FUNCTION ============
def custom_collate_fn(batch):
    """
    Custom collate function to handle PIL Images.
    Returns lists instead of trying to stack PIL Images.
    """
    images = [item[0] for item in batch]
    sample_ids = [item[1] for item in batch]
    success_flags = [item[2] for item in batch]
    return images, sample_ids, success_flags

# ============ HELPER FUNCTION ============
def get_filename_from_url(image_link):
    """
    Extract filename from URL - same logic as download script.
    """
    if isinstance(image_link, str):
        try:
            parsed_url = urllib.parse.urlparse(image_link)
            filename = Path(urllib.parse.unquote(parsed_url.path)).name
            return filename if filename else None
        except Exception:
            return None
    return None

# ============ CUSTOM DATASET ============
class SigLIPImageDataset(Dataset):
    def __init__(self, df, img_dir):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        sample_id = self.df.loc[idx, 'sample_id']
        image_link = self.df.loc[idx, 'image_link']
        
        # Get filename from URL (same as download script)
        filename = get_filename_from_url(image_link)
        
        if filename:
            img_path = os.path.join(self.img_dir, filename)
        else:
            # Fallback to sample_id.jpg if URL parsing fails
            img_path = os.path.join(self.img_dir, f"{sample_id}.jpg")
        
        try:
            img = Image.open(img_path).convert("RGB")
            success = True
        except Exception as e:
            # Return gray placeholder for missing/corrupted images
            img = Image.new("RGB", (224, 224), color="gray")
            success = False
        
        return img, sample_id, success

# ============ CREATE DATALOADER ============
dataset = SigLIPImageDataset(df, IMAGE_FOLDER)
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,  # Keep original order
    num_workers=NUM_WORKERS,
    pin_memory=True,  # Faster GPU transfer
    collate_fn=custom_collate_fn  # Use custom collate function
)

print(f"Created DataLoader with {len(dataloader)} batches (batch_size={BATCH_SIZE})")

# ============ PROCESS ALL IMAGES ============
all_pixel_values = []
all_sample_ids = []
failed_samples = []
successful_count = 0
failed_count = 0

for batch_idx, batch_data in enumerate(tqdm(dataloader, desc="Processing image batches")):
    img_batch, sample_ids, success_flags = batch_data
    
    try:
        # Process batch with SigLIP image processor (returns PyTorch tensors)
        processed = image_processor(
            images=img_batch,
            return_tensors="pt"
        )
        
        # Collect PyTorch tensors
        all_pixel_values.append(processed['pixel_values'])
        all_sample_ids.extend(sample_ids)
        
        # Count successes and failures
        for sid, success in zip(sample_ids, success_flags):
            if success:
                successful_count += 1
            else:
                failed_count += 1
                failed_samples.append(sid)
        
    except Exception as e:
        print(f"\n❌ Error processing batch {batch_idx}: {e}")
        failed_samples.extend(sample_ids)
        failed_count += len(sample_ids)
        continue

# ============ CONCATENATE ALL BATCHES ============
if all_pixel_values:
    final_images = torch.cat(all_pixel_values, dim=0)
    
    print(f"\n✅ Image preprocessing complete!")
    print(f"{'='*60}")
    print(f"  📊 SUMMARY:")
    print(f"  Total images processed: {len(all_sample_ids)}")
    print(f"  ✅ Successfully loaded: {successful_count} ({successful_count/len(all_sample_ids)*100:.1f}%)")
    print(f"  ⚠️  Failed/Missing: {failed_count} ({failed_count/len(all_sample_ids)*100:.1f}%)")
    print(f"  Final tensor shape: {final_images.shape}")
    print(f"  Tensor device: {final_images.device}")
    print(f"{'='*60}")
    
    # ============ SAVE IMAGE FEATURES AS PYTORCH TENSORS ============
    torch.save(final_images, '/kaggle/working/image_pixel_values.pt')
    torch.save(torch.tensor(all_sample_ids), '/kaggle/working/sample_ids_order.pt')
    
    # Save list of failed samples for reference
    if failed_samples:
        with open('/kaggle/working/failed_samples.txt', 'w') as f:
            f.write('\n'.join(map(str, failed_samples)))
        print(f"\n📝 Failed sample IDs saved to: /kaggle/working/failed_samples.txt")
    
    print("\n💾 Saved files:")
    print(f"  - image_pixel_values.pt: {final_images.shape} ({final_images.element_size() * final_images.nelement() / 1024**2:.1f} MB)")
    print(f"  - sample_ids_order.pt: {len(all_sample_ids)} IDs")
    if failed_samples:
        print(f"  - failed_samples.txt: {len(failed_samples)} failed IDs")
    
    print("\n✅ All preprocessing complete!")
    
else:
    print("❌ No images were successfully processed!")

In [ ]:
import os
import warnings
# Suppress TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import torch
from transformers import AutoImageProcessor, LlamaTokenizerFast
from huggingface_hub import hf_hub_download

print("="*60)
print("🔧 FIXED TEXT TOKENIZATION")
print("="*60)

# Paths
CSV_FILE_PATH = '/kaggle/working/image_dataset_for_kaggle/sampled_data.csv'
MODEL_NAME = "google/siglip-base-patch16-224"

# Download tokenizer files directly
print("\nDownloading tokenizer files...")
tokenizer_json = hf_hub_download(repo_id=MODEL_NAME, filename="tokenizer.json")
tokenizer_config = hf_hub_download(repo_id=MODEL_NAME, filename="tokenizer_config.json")
special_tokens = hf_hub_download(repo_id=MODEL_NAME, filename="special_tokens_map.json")

# Load tokenizer directly from files
tokenizer = LlamaTokenizerFast(
    tokenizer_file=tokenizer_json,
    padding_side="right",
    model_max_length=64
)

# Set the pad token (critical for padding)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

print("✅ Tokenizer loaded successfully!")
print(f"  Vocab size: {tokenizer.vocab_size}")
print(f"  EOS token ID: {tokenizer.eos_token_id}")
print(f"  Pad token ID: {tokenizer.pad_token_id}")

# Load data
df = pd.read_csv(CSV_FILE_PATH)
print(f"\n✅ Loaded {len(df)} samples")

# Process text with the tokenizer
texts = df['catalog_content'].astype(str).tolist()

print(f"\nTokenizing {len(texts)} texts...")
inputs_text = tokenizer(
    texts,
    padding="max_length",
    truncation=True,
    max_length=64,
    return_tensors="np"
)

print(f"\n✅ Text preprocessing complete:")
print(f"  input_ids shape: {inputs_text['input_ids'].shape}")
print(f"  attention_mask shape: {inputs_text['attention_mask'].shape}")
print(f"  Token ID range: [{inputs_text['input_ids'].min()}, {inputs_text['input_ids'].max()}]")

# ============ FIX: CLAMP TOKEN IDS TO VALID RANGE ============
vocab_size = 32000  # SigLIP vocab size
max_token_id = inputs_text['input_ids'].max()

if max_token_id >= vocab_size:
    print(f"\n⚠️  Found token ID {max_token_id} >= vocab size {vocab_size}")
    print(f"   Clamping to valid range [0, {vocab_size-1}]...")
    
    # Replace any token ID >= vocab_size with pad_token_id
    inputs_text['input_ids'] = np.where(
        inputs_text['input_ids'] >= vocab_size,
        tokenizer.pad_token_id if tokenizer.pad_token_id < vocab_size else 0,
        inputs_text['input_ids']
    )
    
    print(f"   ✅ Fixed! New range: [{inputs_text['input_ids'].min()}, {inputs_text['input_ids'].max()}]")

# Verify all tokens are valid
assert inputs_text['input_ids'].max() < vocab_size, f"Still have invalid tokens! Max: {inputs_text['input_ids'].max()}"
assert inputs_text['input_ids'].min() >= 0, f"Have negative token IDs! Min: {inputs_text['input_ids'].min()}"

print(f"\n✅ All token IDs verified valid!")

# Save text features
np.save('/kaggle/working/text_input_ids.npy', inputs_text['input_ids'])
np.save('/kaggle/working/text_attention_mask.npy', inputs_text['attention_mask'])

# Also update the global tensor variables
text_input_ids = inputs_text['input_ids']
text_attention_mask = inputs_text['attention_mask']
text_input_ids_tensor = torch.from_numpy(text_input_ids)
text_attention_mask_tensor = torch.from_numpy(text_attention_mask)

print(f"\n💾 Text features saved!")
print(f"  - /kaggle/working/text_input_ids.npy")
print(f"  - /kaggle/working/text_attention_mask.npy")
print(f"\n✅ Ready for SigLIP inference!")

In [ ]:
import os
import warnings
import torch
import numpy as np
from transformers import AutoModel

# Suppress warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'  # Enable synchronous CUDA for better errors
warnings.filterwarnings('ignore')

# Clear any existing CUDA cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("🧹 Cleared CUDA cache")

print("="*60)
print("📥 BLOCK 1: Loading Preprocessed Data")
print("="*60)

# Load PyTorch tensors (images)
pixel_values = torch.load('/kaggle/working/image_pixel_values.pt', map_location='cpu')  # Load to CPU first
sample_ids = torch.load('/kaggle/working/sample_ids_order.pt', map_location='cpu')

# Load numpy arrays (text)
text_input_ids = np.load('/kaggle/working/text_input_ids.npy')
text_attention_mask = np.load('/kaggle/working/text_attention_mask.npy')

# Convert text to PyTorch tensors (on CPU)
text_input_ids_tensor = torch.from_numpy(text_input_ids)
text_attention_mask_tensor = torch.from_numpy(text_attention_mask)

print(f"✅ Data loaded successfully!")
print(f"  Images: {pixel_values.shape} ({pixel_values.dtype}) [CPU]")
print(f"  Text IDs: {text_input_ids_tensor.shape} ({text_input_ids_tensor.dtype}) [CPU]")
print(f"  Attention Mask: {text_attention_mask_tensor.shape}")
print(f"  Sample IDs: {len(sample_ids)}")

# ============================================================
print("\n" + "="*60)
print("🔄 BLOCK 2: Loading SigLIP Model")
print("="*60)

MODEL_NAME = "google/siglip-base-patch16-224"

print(f"Loading model: {MODEL_NAME}")
print("Using FP32 for stability...")

model = AutoModel.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True
)

print(f"✅ Model loaded successfully!")
print(f"  Model type: {type(model).__name__}")
print(f"  Model dtype: {next(model.parameters()).dtype}")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")

# ============================================================
print("\n" + "="*60)
print("🎮 BLOCK 3: GPU Setup")
print("="*60)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"    Memory: {torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB")

# Multi-GPU setup
if torch.cuda.device_count() > 1:
    print(f"\n🔥 Using DataParallel with {torch.cuda.device_count()} GPUs!")
    use_data_parallel = True
else:
    print(f"\n✅ Using single GPU")
    use_data_parallel = False

# Move model to GPU carefully
try:
    if use_data_parallel:
        # For multi-GPU, wrap AFTER moving to cuda
        model = model.cuda()
        model = torch.nn.DataParallel(model)
    else:
        model = model.to(DEVICE)
    
    model.eval()
    print(f"\n✅ Model ready on {DEVICE}")
    print(f"  DataParallel: {use_data_parallel}")
    
except RuntimeError as e:
    print(f"\n❌ Error moving model to GPU: {e}")
    print("\n🔄 Attempting to reset CUDA...")
    torch.cuda.empty_cache()
    
    # Try again with fresh GPU
    if use_data_parallel:
        model = model.cuda()
        model = torch.nn.DataParallel(model)
    else:
        model = model.to(DEVICE)
    
    model.eval()
    print(f"✅ Model moved successfully on retry")

In [ ]:
from tqdm import tqdm

print("\n" + "="*60)
print("🧪 BLOCK 4: Test Single Batch")
print("="*60)

# Test with small batch first
TEST_BATCH_SIZE = 4

print(f"Testing with batch size: {TEST_BATCH_SIZE}")

# Get test batch
test_pixels = pixel_values[:TEST_BATCH_SIZE].to(DEVICE)
test_input_ids = text_input_ids_tensor[:TEST_BATCH_SIZE].to(DEVICE)
test_attention = text_attention_mask_tensor[:TEST_BATCH_SIZE].to(DEVICE)

print(f"  Test pixel batch: {test_pixels.shape} ({test_pixels.dtype})")
print(f"  Test text batch: {test_input_ids.shape} ({test_input_ids.dtype})")
print(f"  Test attention: {test_attention.shape} ({test_attention.dtype})")

# Run inference
with torch.no_grad():
    try:
        if use_data_parallel:
            test_img_emb = model.module.get_image_features(pixel_values=test_pixels)
            test_txt_emb = model.module.get_text_features(
                input_ids=test_input_ids,
                attention_mask=test_attention
            )
        else:
            test_img_emb = model.get_image_features(pixel_values=test_pixels)
            test_txt_emb = model.get_text_features(
                input_ids=test_input_ids,
                attention_mask=test_attention
            )
        
        print(f"\n✅ Test batch successful!")
        print(f"  Image embeddings: {test_img_emb.shape} ({test_img_emb.dtype})")
        print(f"  Text embeddings: {test_txt_emb.shape} ({test_txt_emb.dtype})")
        print(f"\n  Sample embeddings (first 5 dims):")
        print(f"    Image: {test_img_emb[0, :5].cpu().numpy()}")
        print(f"    Text:  {test_txt_emb[0, :5].cpu().numpy()}")
        
    except Exception as e:
        print(f"\n❌ Test batch failed!")
        print(f"  Error: {e}")
        import traceback
        traceback.print_exc()
        raise e

# Clear test tensors
del test_pixels, test_input_ids, test_attention, test_img_emb, test_txt_emb
torch.cuda.empty_cache()
print(f"\n🧹 Test tensors cleared from GPU")

In [ ]:
print("\n" + "="*60)
print("🖼️  BLOCK 5: Extract Image Features")
print("="*60)

INFERENCE_BATCH_SIZE = 128
all_image_embeddings = []

total_batches = (len(pixel_values) + INFERENCE_BATCH_SIZE - 1) // INFERENCE_BATCH_SIZE
print(f"Processing {len(pixel_values)} images in {total_batches} batches")
print(f"Batch size: {INFERENCE_BATCH_SIZE}")

with torch.no_grad():
    for i in tqdm(range(0, len(pixel_values), INFERENCE_BATCH_SIZE), 
                  desc="Image features", total=total_batches):
        
        end_idx = min(i + INFERENCE_BATCH_SIZE, len(pixel_values))
        pixel_batch = pixel_values[i:end_idx].to(DEVICE)
        
        try:
            if use_data_parallel:
                image_embeds = model.module.get_image_features(pixel_values=pixel_batch)
            else:
                image_embeds = model.get_image_features(pixel_values=pixel_batch)
            
            # Convert to CPU and numpy immediately to free GPU memory
            all_image_embeddings.append(image_embeds.cpu().numpy())
            
            # Clear batch from GPU
            del pixel_batch, image_embeds
            
        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"\n⚠️  GPU OOM at batch {i//INFERENCE_BATCH_SIZE}")
                print(f"   Current batch size: {INFERENCE_BATCH_SIZE}")
                print(f"   Try reducing INFERENCE_BATCH_SIZE to 64 or 32")
                torch.cuda.empty_cache()
                raise e
            else:
                raise e
    
    # Clear GPU cache after image processing
    torch.cuda.empty_cache()

# Concatenate all batches
final_image_embeddings = np.vstack(all_image_embeddings)

print(f"\n✅ Image features extracted!")
print(f"  Shape: {final_image_embeddings.shape}")
print(f"  Dtype: {final_image_embeddings.dtype}")
print(f"  Memory: {final_image_embeddings.nbytes / 1024**2:.1f} MB")
print(f"  Value range: [{final_image_embeddings.min():.4f}, {final_image_embeddings.max():.4f}]")

In [ ]:
print("\n" + "="*60)
print("📝 BLOCK 6: Extract Text Features")
print("="*60)

all_text_embeddings = []

print(f"Processing {len(text_input_ids_tensor)} text samples in {total_batches} batches")
print(f"Batch size: {INFERENCE_BATCH_SIZE}")

with torch.no_grad():
    for i in tqdm(range(0, len(text_input_ids_tensor), INFERENCE_BATCH_SIZE), 
                  desc="Text features", total=total_batches):
        
        end_idx = min(i + INFERENCE_BATCH_SIZE, len(text_input_ids_tensor))
        
        input_ids_batch = text_input_ids_tensor[i:end_idx].to(DEVICE)
        attention_batch = text_attention_mask_tensor[i:end_idx].to(DEVICE)
        
        try:
            if use_data_parallel:
                text_embeds = model.module.get_text_features(
                    input_ids=input_ids_batch,
                    attention_mask=attention_batch
                )
            else:
                text_embeds = model.get_text_features(
                    input_ids=input_ids_batch,
                    attention_mask=attention_batch
                )
            
            # Convert to CPU and numpy immediately
            all_text_embeddings.append(text_embeds.cpu().numpy())
            
            # Clear batch from GPU
            del input_ids_batch, attention_batch, text_embeds
            
        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"\n⚠️  GPU OOM at batch {i//INFERENCE_BATCH_SIZE}")
                torch.cuda.empty_cache()
                raise e
            else:
                raise e
    
    # Clear GPU cache
    torch.cuda.empty_cache()

# Concatenate all batches
final_text_embeddings = np.vstack(all_text_embeddings)

print(f"\n✅ Text features extracted!")
print(f"  Shape: {final_text_embeddings.shape}")
print(f"  Dtype: {final_text_embeddings.dtype}")
print(f"  Memory: {final_text_embeddings.nbytes / 1024**2:.1f} MB")
print(f"  Value range: [{final_text_embeddings.min():.4f}, {final_text_embeddings.max():.4f}]")

In [ ]:
print("\n" + "="*60)
print("📊 BLOCK 7: Statistics & Similarity")
print("="*60)

# Image embedding stats
print(f"\n📈 Image Embeddings Statistics:")
print(f"  Mean:  {final_image_embeddings.mean():.6f}")
print(f"  Std:   {final_image_embeddings.std():.6f}")
print(f"  Min:   {final_image_embeddings.min():.6f}")
print(f"  Max:   {final_image_embeddings.max():.6f}")
print(f"  Median: {np.median(final_image_embeddings):.6f}")

# Text embedding stats
print(f"\n📈 Text Embeddings Statistics:")
print(f"  Mean:  {final_text_embeddings.mean():.6f}")
print(f"  Std:   {final_text_embeddings.std():.6f}")
print(f"  Min:   {final_text_embeddings.min():.6f}")
print(f"  Max:   {final_text_embeddings.max():.6f}")
print(f"  Median: {np.median(final_text_embeddings):.6f}")

# Compute similarity
print(f"\n🔗 Computing image-text similarity...")

# L2 Normalize embeddings
image_norms = np.linalg.norm(final_image_embeddings, axis=1, keepdims=True)
text_norms = np.linalg.norm(final_text_embeddings, axis=1, keepdims=True)

image_normalized = final_image_embeddings / (image_norms + 1e-8)
text_normalized = final_text_embeddings / (text_norms + 1e-8)

# Cosine similarity (dot product of normalized vectors)
similarity_scores = np.sum(image_normalized * text_normalized, axis=1)

print(f"\n📈 Image-Text Similarity Scores:")
print(f"  Mean:    {similarity_scores.mean():.6f}")
print(f"  Median:  {np.median(similarity_scores):.6f}")
print(f"  Std:     {similarity_scores.std():.6f}")
print(f"  Min:     {similarity_scores.min():.6f}")
print(f"  Max:     {similarity_scores.max():.6f}")

# Show distribution
print(f"\n  Distribution:")
print(f"    [0.0-0.2]: {np.sum((similarity_scores >= 0.0) & (similarity_scores < 0.2))} samples")
print(f"    [0.2-0.4]: {np.sum((similarity_scores >= 0.2) & (similarity_scores < 0.4))} samples")
print(f"    [0.4-0.6]: {np.sum((similarity_scores >= 0.4) & (similarity_scores < 0.6))} samples")
print(f"    [0.6-0.8]: {np.sum((similarity_scores >= 0.6) & (similarity_scores < 0.8))} samples")
print(f"    [0.8-1.0]: {np.sum((similarity_scores >= 0.8) & (similarity_scores <= 1.0))} samples")

In [ ]:
print("\n" + "="*60)
print("💾 BLOCK 8: Save Embeddings")
print("="*60)

import numpy as np

# Save individual embeddings
print("Saving embeddings to disk...")

np.save('/kaggle/working/siglip_image_embeddings.npy', final_image_embeddings)
np.save('/kaggle/working/siglip_text_embeddings.npy', final_text_embeddings)
np.save('/kaggle/working/siglip_similarity_scores.npy', similarity_scores)

# Save normalized embeddings (useful for similarity searches later)
np.save('/kaggle/working/siglip_image_normalized.npy', image_normalized)
np.save('/kaggle/working/siglip_text_normalized.npy', text_normalized)

# Save combined features (concatenated image + text)
combined_features = np.hstack([final_image_embeddings, final_text_embeddings])
np.save('/kaggle/working/siglip_combined_features.npy', combined_features)

# Save combined features with similarity score as additional feature
combined_with_sim = np.hstack([
    final_image_embeddings,           # [1000, 512]
    final_text_embeddings,            # [1000, 512]
    similarity_scores.reshape(-1, 1)  # [1000, 1]
])
np.save('/kaggle/working/siglip_combined_with_similarity.npy', combined_with_sim)

print(f"\n✅ All embeddings saved successfully!")
print(f"\n{'='*60}")
print(f"📁 SAVED FILES:")
print(f"{'='*60}")
print(f"1. siglip_image_embeddings.npy")
print(f"   Shape: {final_image_embeddings.shape}")
print(f"   Size: {final_image_embeddings.nbytes / 1024**2:.2f} MB")

print(f"\n2. siglip_text_embeddings.npy")
print(f"   Shape: {final_text_embeddings.shape}")
print(f"   Size: {final_text_embeddings.nbytes / 1024**2:.2f} MB")

print(f"\n3. siglip_similarity_scores.npy")
print(f"   Shape: {similarity_scores.shape}")
print(f"   Size: {similarity_scores.nbytes / 1024:.2f} KB")

print(f"\n4. siglip_image_normalized.npy (L2-normalized)")
print(f"   Shape: {image_normalized.shape}")
print(f"   Size: {image_normalized.nbytes / 1024**2:.2f} MB")

print(f"\n5. siglip_text_normalized.npy (L2-normalized)")
print(f"   Shape: {text_normalized.shape}")
print(f"   Size: {text_normalized.nbytes / 1024**2:.2f} MB")

print(f"\n6. siglip_combined_features.npy (image + text)")
print(f"   Shape: {combined_features.shape}")
print(f"   Size: {combined_features.nbytes / 1024**2:.2f} MB")
print(f"   Features: 512 (image) + 512 (text) = 1024 total")

print(f"\n7. siglip_combined_with_similarity.npy (image + text + sim)")
print(f"   Shape: {combined_with_sim.shape}")
print(f"   Size: {combined_with_sim.nbytes / 1024**2:.2f} MB")
print(f"   Features: 512 (image) + 512 (text) + 1 (similarity) = 1025 total")

# Calculate total storage
total_size = (
    final_image_embeddings.nbytes + 
    final_text_embeddings.nbytes + 
    similarity_scores.nbytes +
    image_normalized.nbytes +
    text_normalized.nbytes +
    combined_features.nbytes +
    combined_with_sim.nbytes
) / 1024**2

print(f"\n{'='*60}")
print(f"📦 TOTAL STORAGE: {total_size:.2f} MB")
print(f"{'='*60}")
print(f"\n✅ Ready for downstream tasks!")
print(f"   • XGBoost regression")
print(f"   • Feature analysis")
print(f"   • Similarity search")

In [ ]:
print("\n" + "="*60)
print("🔍 BLOCK 9: Final Verification & Cleanup")
print("="*60)

import pandas as pd

# Load sample IDs
sample_ids_np = sample_ids.numpy() if isinstance(sample_ids, torch.Tensor) else sample_ids

# Load original CSV for verification
df = pd.read_csv('/kaggle/working/image_dataset_for_kaggle/sampled_data.csv')

print(f"\n📊 DATASET SUMMARY:")
print(f"{'='*60}")
print(f"Total samples processed: {len(sample_ids_np)}")
print(f"CSV rows: {len(df)}")
print(f"Match: {'✅ Yes' if len(sample_ids_np) == len(df) else '❌ No'}")

print(f"\n📈 FEATURE SUMMARY:")
print(f"{'='*60}")
print(f"Image embeddings:  {final_image_embeddings.shape}")
print(f"Text embeddings:   {final_text_embeddings.shape}")
print(f"Combined features: {combined_features.shape}")
print(f"Similarity scores: {similarity_scores.shape}")

# Show sample data
print(f"\n🔍 SAMPLE VERIFICATION (First 3 samples):")
print(f"{'='*60}")

for i in range(min(3, len(sample_ids_np))):
    print(f"\n  Sample {i+1}:")
    print(f"    ID: {sample_ids_np[i]}")
    print(f"    Image embedding (first 5): {final_image_embeddings[i, :5]}")
    print(f"    Text embedding (first 5):  {final_text_embeddings[i, :5]}")
    print(f"    Image L2 norm: {np.linalg.norm(final_image_embeddings[i]):.4f}")
    print(f"    Text L2 norm:  {np.linalg.norm(final_text_embeddings[i]):.4f}")
    print(f"    Similarity:    {similarity_scores[i]:.4f}")

# Check for NaN or Inf values
print(f"\n🔍 DATA QUALITY CHECKS:")
print(f"{'='*60}")
print(f"Image embeddings:")
print(f"  NaN values: {np.isnan(final_image_embeddings).sum()}")
print(f"  Inf values: {np.isinf(final_image_embeddings).sum()}")

print(f"\nText embeddings:")
print(f"  NaN values: {np.isnan(final_text_embeddings).sum()}")
print(f"  Inf values: {np.isinf(final_text_embeddings).sum()}")

print(f"\nSimilarity scores:")
print(f"  NaN values: {np.isnan(similarity_scores).sum()}")
print(f"  Inf values: {np.isinf(similarity_scores).sum()}")

if (np.isnan(final_image_embeddings).sum() == 0 and 
    np.isnan(final_text_embeddings).sum() == 0):
    print(f"\n✅ All data is clean (no NaN or Inf values)!")
else:
    print(f"\n⚠️  Warning: Found NaN or Inf values!")

# GPU cleanup
print(f"\n🧹 CLEANUP:")
print(f"{'='*60}")

# Clear model from GPU
del model
if 'test_img_emb' in locals():
    del test_img_emb
if 'test_txt_emb' in locals():
    del test_txt_emb

torch.cuda.empty_cache()

print(f"  ✅ Model removed from GPU")
print(f"  ✅ GPU memory cleared")

# Memory summary
if torch.cuda.is_available():
    print(f"\n💾 GPU MEMORY:")
    for i in range(torch.cuda.device_count()):
        allocated = torch.cuda.memory_allocated(i) / 1024**2
        reserved = torch.cuda.memory_reserved(i) / 1024**2
        print(f"  GPU {i}: Allocated {allocated:.1f} MB, Reserved {reserved:.1f} MB")

print(f"\n{'='*60}")
print(f"✨ SigLIP FEATURE EXTRACTION COMPLETE!")
print(f"{'='*60}")
print(f"\n📂 Your embeddings are saved in:")
print(f"   /kaggle/working/siglip_*.npy")
print(f"\n🎯 Next steps:")
print(f"   1. Load these embeddings")
print(f"   2. Add manual features (quantity, brand, etc.)")
print(f"   3. Train XGBoost for price prediction")
print(f"\n🚀 Ready for ML training!")

In [ ]:
print("="*60)
print("🔗 FINAL: Combine SigLIP Features Only")
print("="*60)

import numpy as np

# You already have these from Blocks 1-9!
siglip_image = np.load('/kaggle/working/siglip_image_embeddings.npy')
siglip_text = np.load('/kaggle/working/siglip_text_embeddings.npy')
siglip_similarity = np.load('/kaggle/working/siglip_similarity_scores.npy')

print(f"✅ All embeddings loaded!")
print(f"\n📊 Feature Dimensions:")
print(f"  SigLIP Image:      {siglip_image.shape}")
print(f"  SigLIP Text:       {siglip_text.shape}")
print(f"  Similarity:        {siglip_similarity.shape}")

# Combine everything
combined_all = np.hstack([
    siglip_image,                      # 512
    siglip_text,                       # 512
    siglip_similarity.reshape(-1, 1)  # 1
])

print(f"\n✅ Combined Features:")
print(f"  Shape: {combined_all.shape}")
print(f"  Total: 512 (image) + 512 (text) + 1 (sim) = 1025 dims")

# Save
np.save('/kaggle/working/features_all_combined.npy', combined_all)

print(f"\n✅ Saved to: features_all_combined.npy")
print(f"  Size: {combined_all.nbytes / 1024**2:.2f} MB")

print(f"\n{'='*60}")
print(f"✅ FEATURE ENGINEERING COMPLETE!")
print(f"{'='*60}")
print(f"\n🎯 You have 1025 features ready for XGBoost:")
print(f"   • 512 from SigLIP image encoder")
print(f"   • 512 from SigLIP text encoder")
print(f"   • 1 similarity score")
print(f"\n🚀 This is enough! SigLIP's text encoder is already excellent.")
print(f"   Ready for XGBoost training!")

In [ ]:
print("="*60)
print("🎯 BLOCK 12: Train XGBoost with SMAPE Metric")
print("="*60)

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ============ SMAPE FUNCTION (EXACT FORMULA) ============
def smape(actual, predicted):
    """
    Symmetric Mean Absolute Percentage Error
    
    Formula: SMAPE = (1/n) * Σ |predicted - actual| / ((|actual| + |predicted|)/2) * 100
    
    Range: 0% to 200%
    Lower is better, 0% = perfect prediction
    """
    numerator = np.abs(predicted - actual)
    denominator = (np.abs(actual) + np.abs(predicted)) / 2.0
    
    # Handle division by zero (when both actual and predicted are 0)
    smape_value = np.where(
        denominator != 0,
        numerator / denominator,
        0.0  # If both are 0, error is 0
    )
    
    return 100.0 * np.mean(smape_value)

# ============ VERIFY SMAPE WITH YOUR EXAMPLE ============
print("\n🔍 Verifying SMAPE calculation:")
actual_test = 100
predicted_test = 120
smape_test = smape(np.array([actual_test]), np.array([predicted_test]))
print(f"  Example: Actual = ${actual_test}, Predicted = ${predicted_test}")
print(f"  SMAPE = |100-120| / ((|100| + |120|)/2) * 100%")
print(f"  SMAPE = 20 / 110 * 100% = 18.18%")
print(f"  Calculated SMAPE: {smape_test:.2f}% ✅")

# ============ LOAD TRAINING DATA ============
print("\n📥 Loading training data...")
X_train_features = np.load('/kaggle/working/features_all_combined.npy')
train_df = pd.read_csv('/kaggle/working/image_dataset_for_kaggle/sampled_data.csv')
y_train = train_df['price'].values

print(f"✅ Data loaded!")
print(f"  Features: {X_train_features.shape}")
print(f"  Samples: {len(y_train)}")

# ============ PRICE STATISTICS ============
print(f"\n📊 Price Statistics:")
print(f"  Mean:   ${y_train.mean():.2f}")
print(f"  Median: ${np.median(y_train):.2f}")
print(f"  Min:    ${y_train.min():.2f}")
print(f"  Max:    ${y_train.max():.2f}")
print(f"  Std:    ${y_train.std():.2f}")

# ============ VALIDATION SPLIT ============
print(f"\n📊 Creating validation split (80/20)...")
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_features, y_train, test_size=0.2, random_state=42
)
print(f"  Training:   {X_tr.shape[0]} samples")
print(f"  Validation: {X_val.shape[0]} samples")

# ============ TRAIN XGBOOST ============
print(f"\n🔄 Training XGBoost model...")

model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=8,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50,
    tree_method='hist'
)

eval_set = [(X_tr, y_tr), (X_val, y_val)]
model.fit(X_tr, y_tr, eval_set=eval_set, verbose=100)

# ============ EVALUATE ============
print(f"\n📈 Evaluating model...")

y_tr_pred = model.predict(X_tr)
y_val_pred = model.predict(X_val)

# Ensure positive predictions
y_tr_pred = np.maximum(y_tr_pred, 0.01)
y_val_pred = np.maximum(y_val_pred, 0.01)

# Calculate ALL metrics
tr_mae = mean_absolute_error(y_tr, y_tr_pred)
val_mae = mean_absolute_error(y_val, y_val_pred)
tr_rmse = np.sqrt(mean_squared_error(y_tr, y_tr_pred))
val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
tr_smape = smape(y_tr, y_tr_pred)
val_smape = smape(y_val, y_val_pred)

print(f"\n{'='*60}")
print(f"📊 VALIDATION RESULTS")
print(f"{'='*60}")
print(f"\nTraining Set:")
print(f"  MAE:   ${tr_mae:.2f}")
print(f"  RMSE:  ${tr_rmse:.2f}")
print(f"  SMAPE: {tr_smape:.2f}%")

print(f"\nValidation Set (This is your expected test performance):")
print(f"  MAE:   ${val_mae:.2f}")
print(f"  RMSE:  ${val_rmse:.2f}")
print(f"  SMAPE: {val_smape:.2f}% ⭐ (YOUR SCORE)")

# ============ RETRAIN ON FULL DATA ============
print(f"\n🔄 Retraining on ALL training data...")
final_model = xgb.XGBRegressor(
    n_estimators=model.best_iteration,
    learning_rate=0.03,
    max_depth=8,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1,
    tree_method='hist'
)

final_model.fit(X_train_features, y_train, verbose=False)
print(f"✅ Final model trained on all {len(y_train)} samples!")

# Final predictions on training set
y_train_final_pred = final_model.predict(X_train_features)
y_train_final_pred = np.maximum(y_train_final_pred, 0.01)
final_train_smape = smape(y_train, y_train_final_pred)

print(f"\nFinal Training SMAPE: {final_train_smape:.2f}%")

# ============ SAVE MODEL ============
final_model.save_model('/kaggle/working/xgboost_final_model.json')
print(f"\n💾 Model saved: xgboost_final_model.json")

# ============ SAMPLE PREDICTIONS WITH SMAPE ============
print(f"\n🔍 Sample Predictions (first 10):")
print(f"{'#':>3s} {'Actual':>10s} {'Predicted':>10s} {'Abs Err':>10s} {'SMAPE':>10s}")
print("-" * 50)

for i in range(min(10, len(y_train))):
    actual = y_train[i]
    pred = y_train_final_pred[i]
    abs_err = abs(actual - pred)
    sample_smape = smape(np.array([actual]), np.array([pred]))
    print(f"{i+1:>3d} ${actual:>8.2f} ${pred:>8.2f} ${abs_err:>8.2f} {sample_smape:>8.2f}%")

print(f"\n{'='*60}")
print(f"✅ TRAINING COMPLETE!")
print(f"{'='*60}")
print(f"\n🎯 Expected Test SMAPE: ~{val_smape:.2f}%")
print(f"   (Based on validation set performance)")
print(f"\n📌 Key Takeaways:")
print(f"   • SMAPE range: 0% (perfect) to 200% (worst)")
print(f"   • Your validation SMAPE: {val_smape:.2f}%")
print(f"   • Lower is better!")
print(f"   • Model ready for test predictions!")

In [ ]:
import pandas as pd
import urllib.parse
from pathlib import Path
import os

print("="*60)
print("📥 BLOCK 14: Download Test Images")
print("="*60)

# Load test CSV
TEST_CSV = '/kaggle/input/sample-testing-test/sample_test.csv'
test_df = pd.read_csv(TEST_CSV)

print(f"✅ Test CSV loaded: {len(test_df)} samples")
print(f"  Columns: {test_df.columns.tolist()}")

# Create output directory
TEST_IMAGE_DIR = '/kaggle/working/test_images'
os.makedirs(TEST_IMAGE_DIR, exist_ok=True)

print(f"\n🚀 Downloading test images...")
print(f"   Using same download logic as training...")

# Use the SAME download code from training (blocks 1-9)
# Download images from test_df['image_link'] to TEST_IMAGE_DIR
# ... (use your previous download script)

print(f"\n✅ Test images downloaded to: {TEST_IMAGE_DIR}")

In [ ]:
import numpy as np
from transformers import AutoImageProcessor, LlamaTokenizerFast
from huggingface_hub import hf_hub_download

print("="*60)
print("📝 BLOCK 15: Preprocess Test Text")
print("="*60)

MODEL_NAME = "google/siglip-base-patch16-224"

# Download tokenizer files
tokenizer_json = hf_hub_download(repo_id=MODEL_NAME, filename="tokenizer.json")
tokenizer_config = hf_hub_download(repo_id=MODEL_NAME, filename="tokenizer_config.json")
special_tokens = hf_hub_download(repo_id=MODEL_NAME, filename="special_tokens_map.json")

tokenizer = LlamaTokenizerFast(
    tokenizer_file=tokenizer_json,
    padding_side="right",
    model_max_length=64
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

print("✅ Tokenizer loaded!")

# Tokenize TEST text
test_texts = test_df['catalog_content'].astype(str).tolist()

inputs_text = tokenizer(
    test_texts,
    padding="max_length",
    truncation=True,
    max_length=64,
    return_tensors="np"
)

# Fix token IDs (clamp to valid range)
vocab_size = 32000
inputs_text['input_ids'] = np.where(
    inputs_text['input_ids'] >= vocab_size,
    tokenizer.pad_token_id if tokenizer.pad_token_id < vocab_size else 0,
    inputs_text['input_ids']
)

print(f"✅ Test text preprocessed: {inputs_text['input_ids'].shape}")

# Save
np.save('/kaggle/working/test_text_input_ids.npy', inputs_text['input_ids'])
np.save('/kaggle/working/test_text_attention_mask.npy', inputs_text['attention_mask'])

print("✅ Saved test text features!")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm

print("="*60)
print("🖼️  BLOCK 16: Preprocess Test Images")
print("="*60)

image_processor = AutoImageProcessor.from_pretrained(MODEL_NAME, use_fast=True)

class TestImageDataset(Dataset):
    def __init__(self, df, img_dir):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        sample_id = self.df.loc[idx, 'sample_id']
        # Use same filename logic as training
        filename = get_filename_from_url(self.df.loc[idx, 'image_link'])
        img_path = os.path.join(self.img_dir, filename)
        
        try:
            img = Image.open(img_path).convert("RGB")
            success = True
        except:
            img = Image.new("RGB", (224, 224), color="gray")
            success = False
        
        return img, sample_id, success

def custom_collate_fn(batch):
    images = [item[0] for item in batch]
    sample_ids = [item[1] for item in batch]
    success_flags = [item[2] for item in batch]
    return images, sample_ids, success_flags

dataset = TestImageDataset(test_df, TEST_IMAGE_DIR)
dataloader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=4, collate_fn=custom_collate_fn)

all_pixel_values = []
all_sample_ids = []

for img_batch, sample_ids, success_flags in tqdm(dataloader, desc="Processing test images"):
    processed = image_processor(images=img_batch, return_tensors="pt")
    all_pixel_values.append(processed['pixel_values'])
    all_sample_ids.extend(sample_ids)

test_images = torch.cat(all_pixel_values, dim=0)

print(f"\n✅ Test images preprocessed: {test_images.shape}")

torch.save(test_images, '/kaggle/working/test_image_pixel_values.pt')
torch.save(torch.tensor(all_sample_ids), '/kaggle/working/test_sample_ids_order.pt')

print("✅ Saved test image features!")

In [ ]:
import torch
from transformers import AutoModel

print("="*60)
print("🔍 BLOCK 17: Extract Test Features with SigLIP")
print("="*60)

# Load preprocessed test data
test_pixel_values = torch.load('/kaggle/working/test_image_pixel_values.pt', map_location='cpu')
test_text_ids = np.load('/kaggle/working/test_text_input_ids.npy')
test_text_mask = np.load('/kaggle/working/test_text_attention_mask.npy')

test_text_ids_tensor = torch.from_numpy(test_text_ids)
test_text_mask_tensor = torch.from_numpy(test_text_mask)

print(f"✅ Test data loaded!")
print(f"  Images: {test_pixel_values.shape}")
print(f"  Text: {test_text_ids_tensor.shape}")

# Load SigLIP model (same as training)
model = AutoModel.from_pretrained("google/siglip-base-patch16-224", torch_dtype=torch.float32)
model = model.cuda() if torch.cuda.is_available() else model
model.eval()

print("✅ SigLIP model loaded!")

# Extract features (same logic as training blocks 5-6)
BATCH_SIZE = 128
test_image_embeddings = []
test_text_embeddings = []

with torch.no_grad():
    # Image features
    for i in tqdm(range(0, len(test_pixel_values), BATCH_SIZE), desc="Test images"):
        batch = test_pixel_values[i:i+BATCH_SIZE].cuda()
        emb = model.get_image_features(pixel_values=batch)
        test_image_embeddings.append(emb.cpu().numpy())
    
    # Text features
    for i in tqdm(range(0, len(test_text_ids_tensor), BATCH_SIZE), desc="Test text"):
        ids_batch = test_text_ids_tensor[i:i+BATCH_SIZE].cuda()
        mask_batch = test_text_mask_tensor[i:i+BATCH_SIZE].cuda()
        emb = model.get_text_features(input_ids=ids_batch, attention_mask=mask_batch)
        test_text_embeddings.append(emb.cpu().numpy())

test_image_emb = np.vstack(test_image_embeddings)
test_text_emb = np.vstack(test_text_embeddings)

print(f"\n✅ Test features extracted!")
print(f"  Image: {test_image_emb.shape}")
print(f"  Text: {test_text_emb.shape}")

In [ ]:
print("="*60)
print("🔗 BLOCK 20: Combine Test Features")
print("="*60)

# Compute similarity (same as training)
image_norms = np.linalg.norm(test_image_emb, axis=1, keepdims=True)
text_norms = np.linalg.norm(test_text_emb, axis=1, keepdims=True)
image_norm = test_image_emb / (image_norms + 1e-8)
text_norm = test_text_emb / (text_norms + 1e-8)
test_similarity = np.sum(image_norm * text_norm, axis=1)

# Combine features (SAME format as training!)
test_features_combined = np.hstack([
    test_image_emb,                      # 512
    test_text_emb,                       # 512
    test_similarity.reshape(-1, 1)       # 1
])

print(f"✅ Test features combined: {test_features_combined.shape}")
print(f"   Expected: ({len(test_df)}, 1025)")

# Save
np.save('/kaggle/working/test_features_all_combined.npy', test_features_combined)
print("✅ Test features saved!")


In [ ]:
print("="*60)
print("📊 BLOCK 22: Calculate Test Set SMAPE")
print("="*60)

import numpy as np
import pandas as pd

# ============ SMAPE FUNCTION ============
def smape(actual, predicted):
    """
    Symmetric Mean Absolute Percentage Error
    
    Formula: SMAPE = (1/n) * Σ |predicted - actual| / ((|actual| + |predicted|)/2) * 100
    
    Returns: SMAPE value (0-200%), lower is better
    """
    numerator = np.abs(predicted - actual)
    denominator = (np.abs(actual) + np.abs(predicted)) / 2.0
    
    # Handle division by zero
    smape_value = np.where(
        denominator != 0,
        numerator / denominator,
        0.0
    )
    
    return 100.0 * np.mean(smape_value)

# ============ VERIFY SMAPE FORMULA ============
print("\n🔍 Verifying SMAPE formula with your example:")
test_actual = 100
test_pred = 120
test_smape = smape(np.array([test_actual]), np.array([test_pred]))
print(f"  Actual = ${test_actual}, Predicted = ${test_pred}")
print(f"  Expected SMAPE: 18.18%")
print(f"  Calculated SMAPE: {test_smape:.2f}% ✅")

# ============ LOAD GROUND TRUTH ============
print("\n📥 Loading ground truth prices...")

GROUND_TRUTH_CSV = '/kaggle/input/sample-test-output/sample_test_out.csv'
ground_truth_df = pd.read_csv(GROUND_TRUTH_CSV)

print(f"✅ Ground truth loaded: {len(ground_truth_df)} samples")
print(f"  Columns: {ground_truth_df.columns.tolist()}")

# ============ LOAD YOUR PREDICTIONS ============
print("\n📥 Loading your predictions...")

YOUR_PREDICTIONS_CSV = '/kaggle/working/submission.csv'
predictions_df = pd.read_csv(YOUR_PREDICTIONS_CSV)

print(f"✅ Predictions loaded: {len(predictions_df)} samples")

# ============ MERGE AND ALIGN ============
print("\n🔗 Aligning predictions with ground truth...")

# Merge on sample_id to ensure proper alignment
merged_df = ground_truth_df.merge(
    predictions_df,
    on='sample_id',
    suffixes=('_actual', '_predicted')
)

print(f"✅ Merged: {len(merged_df)} samples")

# Check for missing samples
missing_in_predictions = len(ground_truth_df) - len(merged_df)
if missing_in_predictions > 0:
    print(f"⚠️  WARNING: {missing_in_predictions} samples missing in predictions!")

# Extract aligned prices
actual_prices = merged_df['price_actual'].values
predicted_prices = merged_df['price_predicted'].values

# ============ CALCULATE SMAPE ============
test_smape = smape(actual_prices, predicted_prices)

print(f"\n{'='*60}")
print(f"🎯 TEST SET RESULTS")
print(f"{'='*60}")
print(f"\n  📊 TEST SMAPE: {test_smape:.4f}%")
print(f"\n  Interpretation:")
if test_smape < 10:
    print(f"  🏆⭐⭐⭐ EXCELLENT! (< 10%)")
elif test_smape < 15:
    print(f"  ⭐⭐ VERY GOOD! (10-15%)")
elif test_smape < 20:
    print(f"  ⭐ GOOD (15-20%)")
elif test_smape < 30:
    print(f"  ✓ Decent (20-30%)")
else:
    print(f"  ⚠️  Needs improvement (> 30%)")

# ============ ADDITIONAL METRICS ============
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(actual_prices, predicted_prices)
rmse = np.sqrt(mean_squared_error(actual_prices, predicted_prices))
r2 = r2_score(actual_prices, predicted_prices)

print(f"\n  Additional Metrics:")
print(f"    MAE:  ${mae:.2f}")
print(f"    RMSE: ${rmse:.2f}")
print(f"    R²:   {r2:.4f}")

# ============ PRICE STATISTICS ============
print(f"\n📊 Price Statistics:")
print(f"\n  Actual Prices (Ground Truth):")
print(f"    Mean:   ${actual_prices.mean():.2f}")
print(f"    Median: ${np.median(actual_prices):.2f}")
print(f"    Min:    ${actual_prices.min():.2f}")
print(f"    Max:    ${actual_prices.max():.2f}")
print(f"    Std:    ${actual_prices.std():.2f}")

print(f"\n  Your Predicted Prices:")
print(f"    Mean:   ${predicted_prices.mean():.2f}")
print(f"    Median: ${np.median(predicted_prices):.2f}")
print(f"    Min:    ${predicted_prices.min():.2f}")
print(f"    Max:    ${predicted_prices.max():.2f}")
print(f"    Std:    ${predicted_prices.std():.2f}")

# ============ ERROR ANALYSIS ============
errors = predicted_prices - actual_prices
abs_errors = np.abs(errors)
pct_errors = (errors / actual_prices) * 100

print(f"\n📈 Error Distribution:")
print(f"  Mean Error:       ${errors.mean():.2f}")
print(f"  Mean Abs Error:   ${abs_errors.mean():.2f}")
print(f"  Median Abs Error: ${np.median(abs_errors):.2f}")
print(f"  Max Overestimate: ${errors.max():.2f}")
print(f"  Max Underestimate: ${errors.min():.2f}")

print(f"\n  Percentage Errors:")
print(f"    Mean:   {pct_errors.mean():.2f}%")
print(f"    Median: {np.median(pct_errors):.2f}%")
within_10 = np.sum(np.abs(pct_errors) <= 10)
within_20 = np.sum(np.abs(pct_errors) <= 20)
within_30 = np.sum(np.abs(pct_errors) <= 30)
print(f"    Within ±10%: {within_10} samples ({within_10/len(pct_errors)*100:.1f}%)")
print(f"    Within ±20%: {within_20} samples ({within_20/len(pct_errors)*100:.1f}%)")
print(f"    Within ±30%: {within_30} samples ({within_30/len(pct_errors)*100:.1f}%)")

# ============ DETAILED SAMPLE ANALYSIS ============
print(f"\n🔍 Detailed Sample-by-Sample Analysis (First 20):")
print(f"{'Sample ID':>10s} {'Actual':>10s} {'Predicted':>10s} {'Error':>10s} {'Error%':>10s} {'SMAPE':>10s}")
print("-" * 75)

for i in range(min(20, len(merged_df))):
    sample_id = merged_df.iloc[i]['sample_id']
    actual = actual_prices[i]
    pred = predicted_prices[i]
    error = pred - actual
    error_pct = (error / actual) * 100 if actual > 0 else 0
    sample_smape = smape(np.array([actual]), np.array([pred]))
    
    print(f"{sample_id:>10} ${actual:>8.2f} ${pred:>8.2f} ${error:>8.2f} {error_pct:>8.1f}% {sample_smape:>8.2f}%")

# ============ BEST/WORST PREDICTIONS ============
smape_per_sample = np.array([smape(np.array([a]), np.array([p])) 
                              for a, p in zip(actual_prices, predicted_prices)])

print(f"\n❌ Top 5 WORST Predictions (Highest SMAPE):")
worst_indices = np.argsort(smape_per_sample)[-5:][::-1]
print(f"{'Sample ID':>10s} {'Actual':>10s} {'Predicted':>10s} {'SMAPE':>10s}")
print("-" * 50)
for idx in worst_indices:
    sample_id = merged_df.iloc[idx]['sample_id']
    print(f"{sample_id:>10} ${actual_prices[idx]:>8.2f} ${predicted_prices[idx]:>8.2f} {smape_per_sample[idx]:>8.2f}%")

print(f"\n✅ Top 5 BEST Predictions (Lowest SMAPE):")
best_indices = np.argsort(smape_per_sample)[:5]
print(f"{'Sample ID':>10s} {'Actual':>10s} {'Predicted':>10s} {'SMAPE':>10s}")
print("-" * 50)
for idx in best_indices:
    sample_id = merged_df.iloc[idx]['sample_id']
    print(f"{sample_id:>10} ${actual_prices[idx]:>8.2f} ${predicted_prices[idx]:>8.2f} {smape_per_sample[idx]:>8.2f}%")

# ============ SAVE DETAILED RESULTS ============
print(f"\n💾 Saving detailed results...")

merged_df['error'] = errors
merged_df['abs_error'] = abs_errors
merged_df['error_pct'] = pct_errors
merged_df['smape'] = smape_per_sample

merged_df.to_csv('/kaggle/working/test_results_detailed.csv', index=False)
print(f"  ✅ Saved: test_results_detailed.csv")

# Save summary
summary = pd.DataFrame([{
    'test_smape': test_smape,
    'mae': mae,
    'rmse': rmse,
    'r2': r2,
    'mean_error': errors.mean(),
    'median_error': np.median(errors),
    'within_10pct': within_10,
    'within_20pct': within_20,
    'within_30pct': within_30,
    'total_samples': len(actual_prices)
}])

summary.to_csv('/kaggle/working/test_summary.csv', index=False)
print(f"  ✅ Saved: test_summary.csv")

print(f"\n{'='*60}")
print(f"✅ TEST EVALUATION COMPLETE!")
print(f"{'='*60}")
print(f"\n🎯 YOUR FINAL TEST SMAPE: {test_smape:.4f}%")
print(f"   (Lower is better, 0% = perfect prediction)")
print(f"\n📁 Saved files:")
print(f"   - test_results_detailed.csv (per-sample results)")
print(f"   - test_summary.csv (overall metrics)")